# Cathay Bank ML Models
## Customer Churn Prediction, Credit Risk Scoring & Customer Lifetime Value

This notebook trains three ML models for the Cathay Bank Intelligence Agent:
1. **Customer Churn Prediction** - Identify customers likely to close accounts
2. **Credit Risk Scoring** - Predict loan default probability
3. **Customer Lifetime Value (LTV)** - Estimate long-term customer value

All models are registered in the Snowflake Model Registry for use by the agent.

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

session = get_active_session()
print('Session established')

In [ ]:
%%sql -r customer_features
SELECT
    c.CUSTOMER_ID,
    c.CREDIT_SCORE,
    c.ANNUAL_INCOME,
    DATEDIFF('year', c.DATE_OF_BIRTH, CURRENT_DATE()) AS AGE,
    DATEDIFF('year', c.CUSTOMER_SINCE, CURRENT_DATE()) AS TENURE_YEARS,
    c.CUSTOMER_SEGMENT,
    c.EMPLOYMENT_STATUS,
    c.PREFERRED_LANGUAGE,
    c.RISK_RATING,
    c.IS_ACTIVE,
    COALESCE(a.ACCOUNT_COUNT, 0) AS ACCOUNT_COUNT,
    COALESCE(a.TOTAL_BALANCE, 0) AS TOTAL_BALANCE,
    COALESCE(a.AVG_BALANCE, 0) AS AVG_BALANCE,
    COALESCE(t.TXN_COUNT_90D, 0) AS TXN_COUNT_90D,
    COALESCE(t.TOTAL_DEBITS_90D, 0) AS TOTAL_DEBITS_90D,
    COALESCE(t.TOTAL_CREDITS_90D, 0) AS TOTAL_CREDITS_90D,
    COALESCE(l.LOAN_COUNT, 0) AS LOAN_COUNT,
    COALESCE(l.TOTAL_LOAN_BALANCE, 0) AS TOTAL_LOAN_BALANCE,
    COALESCE(s.SUPPORT_CASES, 0) AS SUPPORT_CASES,
    COALESCE(s.AVG_SATISFACTION, 3.0) AS AVG_SATISFACTION
FROM CATHAY_BANK_DB.RAW.CUSTOMERS c
LEFT JOIN (
    SELECT CUSTOMER_ID, COUNT(*) AS ACCOUNT_COUNT,
           SUM(CURRENT_BALANCE) AS TOTAL_BALANCE, AVG(CURRENT_BALANCE) AS AVG_BALANCE
    FROM CATHAY_BANK_DB.RAW.ACCOUNTS WHERE STATUS = 'ACTIVE' GROUP BY CUSTOMER_ID
) a ON c.CUSTOMER_ID = a.CUSTOMER_ID
LEFT JOIN (
    SELECT ac.CUSTOMER_ID, COUNT(*) AS TXN_COUNT_90D,
           SUM(CASE WHEN t.AMOUNT < 0 THEN ABS(t.AMOUNT) ELSE 0 END) AS TOTAL_DEBITS_90D,
           SUM(CASE WHEN t.AMOUNT > 0 THEN t.AMOUNT ELSE 0 END) AS TOTAL_CREDITS_90D
    FROM CATHAY_BANK_DB.RAW.TRANSACTIONS t
    JOIN CATHAY_BANK_DB.RAW.ACCOUNTS ac ON t.ACCOUNT_ID = ac.ACCOUNT_ID
    WHERE t.TRANSACTION_DATE >= DATEADD('day', -90, CURRENT_TIMESTAMP())
    GROUP BY ac.CUSTOMER_ID
) t ON c.CUSTOMER_ID = t.CUSTOMER_ID
LEFT JOIN (
    SELECT CUSTOMER_ID, COUNT(*) AS LOAN_COUNT, SUM(CURRENT_BALANCE) AS TOTAL_LOAN_BALANCE
    FROM CATHAY_BANK_DB.RAW.LOANS WHERE STATUS = 'ACTIVE' GROUP BY CUSTOMER_ID
) l ON c.CUSTOMER_ID = l.CUSTOMER_ID
LEFT JOIN (
    SELECT CUSTOMER_ID, COUNT(*) AS SUPPORT_CASES, AVG(SATISFACTION_SCORE) AS AVG_SATISFACTION
    FROM CATHAY_BANK_DB.RAW.SUPPORT_CASES GROUP BY CUSTOMER_ID
) s ON c.CUSTOMER_ID = s.CUSTOMER_ID

In [ ]:
df = customer_features.copy()
print(f'Customer dataset: {df.shape[0]} rows, {df.shape[1]} columns')
print(f'\nChurn rate (IS_ACTIVE=False): {(~df["IS_ACTIVE"]).mean():.2%}')
print(f'\nFeature summary:')
df.describe().round(2)

In [ ]:
le_segment = LabelEncoder()
le_employment = LabelEncoder()
le_language = LabelEncoder()
le_risk = LabelEncoder()

df['SEGMENT_ENC'] = le_segment.fit_transform(df['CUSTOMER_SEGMENT'].fillna('Standard'))
df['EMPLOYMENT_ENC'] = le_employment.fit_transform(df['EMPLOYMENT_STATUS'].fillna('Employed'))
df['LANGUAGE_ENC'] = le_language.fit_transform(df['PREFERRED_LANGUAGE'].fillna('English'))
df['RISK_ENC'] = le_risk.fit_transform(df['RISK_RATING'].fillna('Medium'))

churn_features = ['CREDIT_SCORE', 'ANNUAL_INCOME', 'AGE', 'TENURE_YEARS',
                  'ACCOUNT_COUNT', 'TOTAL_BALANCE', 'AVG_BALANCE',
                  'TXN_COUNT_90D', 'TOTAL_DEBITS_90D', 'TOTAL_CREDITS_90D',
                  'LOAN_COUNT', 'TOTAL_LOAN_BALANCE', 'SUPPORT_CASES',
                  'AVG_SATISFACTION', 'SEGMENT_ENC', 'EMPLOYMENT_ENC',
                  'LANGUAGE_ENC', 'RISK_ENC']

X_churn = df[churn_features].fillna(0)
y_churn = (~df['IS_ACTIVE']).astype(int)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_churn, y_churn, test_size=0.2, random_state=42, stratify=y_churn)
print(f'Training set: {X_train_c.shape[0]} samples')
print(f'Test set: {X_test_c.shape[0]} samples')
print(f'Churn rate in train: {y_train_c.mean():.2%}')

In [ ]:
churn_model = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    random_state=42
)
churn_model.fit(X_train_c, y_train_c)

y_pred_c = churn_model.predict(X_test_c)
y_proba_c = churn_model.predict_proba(X_test_c)[:, 1]

churn_metrics = {
    'accuracy': round(accuracy_score(y_test_c, y_pred_c), 4),
    'precision': round(precision_score(y_test_c, y_pred_c, zero_division=0), 4),
    'recall': round(recall_score(y_test_c, y_pred_c, zero_division=0), 4),
    'f1_score': round(f1_score(y_test_c, y_pred_c, zero_division=0), 4),
    'roc_auc': round(roc_auc_score(y_test_c, y_proba_c), 4)
}
print('=== Churn Model Performance ===')
for k, v in churn_metrics.items():
    print(f'  {k}: {v}')

feature_importance = pd.DataFrame({
    'feature': churn_features,
    'importance': churn_model.feature_importances_
}).sort_values('importance', ascending=False)
print('\nTop 10 Features:')
print(feature_importance.head(10).to_string(index=False))

In [ ]:
%%sql -r loan_features
SELECT
    l.LOAN_ID,
    l.LOAN_TYPE,
    l.ORIGINAL_AMOUNT,
    l.CURRENT_BALANCE,
    l.INTEREST_RATE,
    l.TERM_MONTHS,
    l.MONTHLY_PAYMENT,
    DATEDIFF('month', l.ORIGINATION_DATE, CURRENT_DATE()) AS MONTHS_SINCE_ORIGINATION,
    COALESCE(l.LTV_RATIO, 0) AS LTV_RATIO,
    COALESCE(l.DTI_RATIO, 0) AS DTI_RATIO,
    l.DAYS_PAST_DUE,
    CASE WHEN l.STATUS = 'DEFAULT' THEN 1 ELSE 0 END AS IS_DEFAULT,
    c.CREDIT_SCORE,
    c.ANNUAL_INCOME,
    DATEDIFF('year', c.DATE_OF_BIRTH, CURRENT_DATE()) AS BORROWER_AGE,
    c.EMPLOYMENT_STATUS,
    COALESCE(p.LATE_PAYMENT_COUNT, 0) AS LATE_PAYMENT_COUNT,
    COALESCE(p.TOTAL_LATE_FEES, 0) AS TOTAL_LATE_FEES
FROM CATHAY_BANK_DB.RAW.LOANS l
JOIN CATHAY_BANK_DB.RAW.CUSTOMERS c ON l.CUSTOMER_ID = c.CUSTOMER_ID
LEFT JOIN (
    SELECT LOAN_ID, SUM(CASE WHEN STATUS = 'Late' THEN 1 ELSE 0 END) AS LATE_PAYMENT_COUNT,
           SUM(LATE_FEE) AS TOTAL_LATE_FEES
    FROM CATHAY_BANK_DB.RAW.LOAN_PAYMENTS GROUP BY LOAN_ID
) p ON l.LOAN_ID = p.LOAN_ID

In [ ]:
df_loan = loan_features.copy()
print(f'Loan dataset: {df_loan.shape[0]} rows, {df_loan.shape[1]} columns')
print(f'Default rate: {df_loan["IS_DEFAULT"].mean():.2%}')

le_loan_type = LabelEncoder()
le_empl = LabelEncoder()
df_loan['LOAN_TYPE_ENC'] = le_loan_type.fit_transform(df_loan['LOAN_TYPE'].fillna('Personal Loan'))
df_loan['EMPLOYMENT_ENC'] = le_empl.fit_transform(df_loan['EMPLOYMENT_STATUS'].fillna('Employed'))

credit_features = ['ORIGINAL_AMOUNT', 'CURRENT_BALANCE', 'INTEREST_RATE', 'TERM_MONTHS',
                   'MONTHLY_PAYMENT', 'MONTHS_SINCE_ORIGINATION', 'LTV_RATIO', 'DTI_RATIO',
                   'DAYS_PAST_DUE', 'CREDIT_SCORE', 'ANNUAL_INCOME', 'BORROWER_AGE',
                   'LATE_PAYMENT_COUNT', 'TOTAL_LATE_FEES', 'LOAN_TYPE_ENC', 'EMPLOYMENT_ENC']

X_credit = df_loan[credit_features].fillna(0)
y_credit = df_loan['IS_DEFAULT']

X_train_cr, X_test_cr, y_train_cr, y_test_cr = train_test_split(X_credit, y_credit, test_size=0.2, random_state=42, stratify=y_credit)
print(f'Training: {X_train_cr.shape[0]}, Test: {X_test_cr.shape[0]}')

In [ ]:
credit_model = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    random_state=42
)
credit_model.fit(X_train_cr, y_train_cr)

y_pred_cr = credit_model.predict(X_test_cr)
y_proba_cr = credit_model.predict_proba(X_test_cr)[:, 1]

credit_metrics = {
    'accuracy': round(accuracy_score(y_test_cr, y_pred_cr), 4),
    'precision': round(precision_score(y_test_cr, y_pred_cr, zero_division=0), 4),
    'recall': round(recall_score(y_test_cr, y_pred_cr, zero_division=0), 4),
    'f1_score': round(f1_score(y_test_cr, y_pred_cr, zero_division=0), 4),
    'roc_auc': round(roc_auc_score(y_test_cr, y_proba_cr), 4)
}
print('=== Credit Risk Model Performance ===')
for k, v in credit_metrics.items():
    print(f'  {k}: {v}')

credit_importance = pd.DataFrame({
    'feature': credit_features,
    'importance': credit_model.feature_importances_
}).sort_values('importance', ascending=False)
print('\nTop 10 Features:')
print(credit_importance.head(10).to_string(index=False))

In [ ]:
df['TOTAL_RELATIONSHIP_VALUE'] = df['TOTAL_BALANCE'].fillna(0) + df['TOTAL_LOAN_BALANCE'].fillna(0)

ltv_features = ['CREDIT_SCORE', 'ANNUAL_INCOME', 'AGE', 'TENURE_YEARS',
                'ACCOUNT_COUNT', 'TOTAL_BALANCE', 'TXN_COUNT_90D',
                'TOTAL_CREDITS_90D', 'LOAN_COUNT', 'TOTAL_LOAN_BALANCE',
                'SUPPORT_CASES', 'AVG_SATISFACTION', 'SEGMENT_ENC',
                'EMPLOYMENT_ENC', 'RISK_ENC']

X_ltv = df[ltv_features].fillna(0)
y_ltv = df['TOTAL_RELATIONSHIP_VALUE'].fillna(0)

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_ltv, y_ltv, test_size=0.2, random_state=42)

ltv_model = GradientBoostingRegressor(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    random_state=42
)
ltv_model.fit(X_train_l, y_train_l)

y_pred_l = ltv_model.predict(X_test_l)

ltv_metrics = {
    'r2_score': round(r2_score(y_test_l, y_pred_l), 4),
    'mae': round(mean_absolute_error(y_test_l, y_pred_l), 2),
    'median_predicted_ltv': round(np.median(y_pred_l), 2),
    'mean_predicted_ltv': round(np.mean(y_pred_l), 2)
}
print('=== Customer LTV Model Performance ===')
for k, v in ltv_metrics.items():
    print(f'  {k}: {v}')

In [ ]:
from snowflake.ml.registry import Registry

reg = Registry(session=session, database_name='CATHAY_BANK_DB', schema_name='ML')

reg.log_model(
    model=churn_model,
    model_name='cathay_churn_model',
    version_name='v1',
    sample_input_data=X_train_c.head(10),
    metrics=churn_metrics,
    comment='Cathay Bank customer churn prediction model using GradientBoosting'
)
print('Churn model registered successfully')

reg.log_model(
    model=credit_model,
    model_name='cathay_credit_risk_model',
    version_name='v1',
    sample_input_data=X_train_cr.head(10),
    metrics=credit_metrics,
    comment='Cathay Bank credit risk / default prediction model using GradientBoosting'
)
print('Credit risk model registered successfully')

reg.log_model(
    model=ltv_model,
    model_name='cathay_ltv_model',
    version_name='v1',
    sample_input_data=X_train_l.head(10),
    metrics=ltv_metrics,
    comment='Cathay Bank customer lifetime value prediction model using GradientBoosting'
)
print('LTV model registered successfully')

print('\n=== All Models Registered ===')
for m in reg.show_models():
    print(f'  - {m}')

In [ ]:
print('=' * 60)
print('CATHAY BANK ML MODEL TRAINING SUMMARY')
print('=' * 60)

print('\n1. CHURN PREDICTION MODEL')
print(f'   Model: GradientBoostingClassifier')
print(f'   Registry: CATHAY_BANK_DB.ML.cathay_churn_model/v1')
for k, v in churn_metrics.items():
    print(f'   {k}: {v}')

print('\n2. CREDIT RISK MODEL')
print(f'   Model: GradientBoostingClassifier')
print(f'   Registry: CATHAY_BANK_DB.ML.cathay_credit_risk_model/v1')
for k, v in credit_metrics.items():
    print(f'   {k}: {v}')

print('\n3. CUSTOMER LTV MODEL')
print(f'   Model: GradientBoostingRegressor')
print(f'   Registry: CATHAY_BANK_DB.ML.cathay_ltv_model/v1')
for k, v in ltv_metrics.items():
    print(f'   {k}: {v}')

print('\n' + '=' * 60)
print('All models registered and ready for agent consumption.')
print('=' * 60)